# 03 — MVP Audit + Analyze + Visualize

Consumes the 600 generated images from `02_databricks_mvp_run` (already on `/Volumes/ds_work/alenj00/cap_cache/runs/mvp/generated/`) and produces:

1. **Audit predictions** (DeepFace, gender + age) — `runs/mvp/audit/predictions.parquet`
2. **Statistical analysis** (counterfactual flip rate, intersectional error tables) — `runs/mvp/analysis/`
3. **Publication figures + interactive dashboard** — `runs/mvp/viz/`

**Cluster:** same single-user cluster (`6106-192556-lj0uddmy`). DeepFace runs on the L4 driver — workers idle this time.

**Expected wall:** ~30–50 min (audit dominates).

## 1. Install cap

In [ ]:
%pip install /Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline

In [ ]:
dbutils.library.restartPython()

## 2. Verify generated images present

In [ ]:
from pathlib import Path
import pandas as pd

GENERATED = Path("/Volumes/ds_work/alenj00/cap_cache/runs/mvp/generated")
pngs = list(GENERATED.glob("*.png"))
manifest = GENERATED / "manifest.parquet"
print(f"PNGs in {GENERATED}: {len(pngs)}")
assert len(pngs) >= 600, f"Expected ≥600 PNGs, found {len(pngs)} — generation step incomplete"
assert manifest.exists(), f"Missing {manifest} — generation step never finished"
df = pd.read_parquet(manifest)
print(f"Manifest rows: {len(df)}")
print("By axis combo:")
if {"axis_skin_tone", "axis_gender"}.issubset(df.columns):
    print(df.groupby(["axis_skin_tone", "axis_gender"]).size().unstack(fill_value=0))

## 3. Audit (DeepFace × gender + age × 600 images)

DeepFace will lazy-download its detector + recognition models on first call (~few hundred MB). Predictions stream into `predictions.parquet`.

In [ ]:
import time
from click.testing import CliRunner
from cap.cli.audit import main as audit_cli

REPO = "/Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline"

t0 = time.time()
runner = CliRunner()
result = runner.invoke(
    audit_cli,
    ["--config", f"{REPO}/configs/mvp.yaml"],
    catch_exceptions=False,
    standalone_mode=False,
)
wall = time.time() - t0
print(f"\nAudit wall: {wall:.0f}s ({wall/60:.1f} min)")
print("CLI output:")
print(result.output[:3000])

## 4. Analyze

In [ ]:
from cap.cli.analyze import main as analyze_cli

t0 = time.time()
result = runner.invoke(
    analyze_cli,
    ["--config", f"{REPO}/configs/mvp.yaml"],
    catch_exceptions=False,
    standalone_mode=False,
)
wall = time.time() - t0
print(f"Analyze wall: {wall:.0f}s")
print("CLI output:")
print(result.output[:3000])

print("\nAnalysis outputs:")
for p in sorted(Path("/Volumes/ds_work/alenj00/cap_cache/runs/mvp/analysis").glob("*")):
    print(f"  {p.name} ({p.stat().st_size} bytes)")

## 5. Visualize

In [ ]:
from cap.cli.visualize import main as visualize_cli

t0 = time.time()
result = runner.invoke(
    visualize_cli,
    ["--config", f"{REPO}/configs/mvp.yaml"],
    catch_exceptions=False,
    standalone_mode=False,
)
wall = time.time() - t0
print(f"Visualize wall: {wall:.0f}s")
print("CLI output:")
print(result.output[:3000])

print("\nFigures:")
VIZ = Path("/Volumes/ds_work/alenj00/cap_cache/runs/mvp/viz")
for p in sorted(VIZ.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(VIZ)} ({p.stat().st_size} bytes)")

## 6. Quick visual preview

In [ ]:
from PIL import Image
import io

# Sample one generated counterfactual to confirm output shape
sample_png = sorted(GENERATED.glob("*.png"))[0]
print(f"Sample: {sample_png.name}")
display(Image.open(sample_png))

# Sample audit predictions
audit_path = Path("/Volumes/ds_work/alenj00/cap_cache/runs/mvp/audit/predictions.parquet")
if audit_path.exists():
    pdf = pd.read_parquet(audit_path)
    print(f"\nAudit predictions: {len(pdf)} rows, {pdf['auditor'].nunique()} auditor(s), {pdf['task'].nunique()} task(s)")
    print(pdf[['seed_identity_id', 'auditor', 'task', 'prediction', 'confidence']].head(10))